# Remote Work and Urban Housing Prices: Exploratory Data Analysis
Team 10 — Sai Krishna Varshith Danda, Gayathri Kota, Eshitha Chowdary Kaka, Zain Khan

*Skeleton. Section 1 pulls its tables from the pipeline's data-quality reports so the text and the data cannot drift apart; Section 2 is the figure plan, with the question each figure answers. Replace each TODO with a figure and a short interpretation paragraph.*

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN, LOGS = ROOT / "data" / "clean", ROOT / "outputs" / "logs"
FIGS = ROOT / "outputs" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)

def log(name):  # show a pipeline report inline
    display(Markdown((LOGS / name).read_text()))

In [ ]:
# Data (all produced by src/run_pipeline.py)
panel  = pd.read_parquet(CLEAN / "zhvi_analysis_panel.parquet")        # ZIP x month: zhvi, log_zhvi, yoy_pct, index_dec2019, period, in_window
zg     = pd.read_csv(CLEAN / "zip_growth.csv", dtype={"zip": str})      # one row per ZIP: December YoY, cumulative growth, growth relative to metro
dist   = pd.read_csv(CLEAN / "zip_distance.csv", dtype={"zip": str})    # distance to city hall / nearest center / ZCTA centroid, bands
flags  = pd.read_csv(CLEAN / "zip_flags.csv", dtype={"zip": str})       # data-quality flags per ZIP
acs    = pd.read_csv(CLEAN / "zip_acs.csv", dtype={"zip": str})         # ACS 2015-2019 (_2019) and 2019-2023 (_2023) features
zori   = pd.read_parquet(CLEAN / "zori_analysis_panel.parquet")
zorig  = pd.read_csv(CLEAN / "zori_growth.csv", dtype={"zip": str})     # fixed sample: ZORI at Dec 2019 and Dec 2025
zcta   = gpd.read_file(CLEAN / "zcta_boundaries.gpkg")                  # polygons for the 1,099 ZIPs
print(panel.shape, zg.shape, zcta.shape)

In [ ]:
# Plot conventions (set once so every figure matches): fixed color per metro, one font scale, one color map per quantity.
METROS = ["Phoenix", "Austin", "Atlanta", "Dallas", "Tampa", "Charlotte", "Nashville", "Las Vegas"]
PERIODS = ["pre-2020", "2020-2022", "2023-2025"]
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 10,
                     "figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False, "axes.spines.right": False})
# TODO: choose the metro palette (colorblind-safe, 8 categories) and the sequential/diverging maps; see the dataviz guidance.

## 1. Data sources, preparation, and cleaning
Five public sources, joined at ZIP / ZCTA level for eight Sun Belt metros. Full provenance (URL, release, download date, size) is in `docs/SOURCES.md`. Everything below is produced by `python src/run_pipeline.py`.

### 1.1 Sources
TODO: one short paragraph per source — what it is, why we use it, what the unit is.
- **Zillow ZHVI** (outcome): ZIP-level, all homes, mid-tier, smoothed, seasonally adjusted, monthly 2000–2026.
- **Zillow ZORI** (secondary outcome): rents, same geography, ~8,400 ZIPs nationally.
- **ACS 5-year**, 2015–2019 (pre-pandemic baseline) and 2019–2023 (current composition, descriptive only): B08006, C24010, B19013, B25024, B01003 at ZCTA level via the Census API.
- **Census Gazetteer** (ZCTA centroids), **CBSA delineation** (county crosswalk), **cartographic ZCTA boundaries** (maps).
- **City-hall coordinates**, hand-compiled with a source recorded for each of the eight metros (plus a second center for Dallas–Fort Worth and Tampa–St. Petersburg).

### 1.2 What we did to the data and what we found while doing it
Highlights to write up (each is backed by a report below): metro membership checked two ways; ZIP→ZCTA match rates; ZHVI gaps; ZORI coverage limits; a margin-of-error reliability flag on the pre-pandemic work-from-home share; two center definitions plus a second center for two-center metros; and one consolidated finding that most data problems sit at the periphery.

In [ ]:
log('zhvi_quality_report.md')

In [ ]:
log('metro_crosswalk_report.md')

In [ ]:
log('distance_report.md')

In [ ]:
log('acs_report.md')

In [ ]:
log('zori_coverage_report.md')

In [ ]:
log('boundaries_report.md')

In [ ]:
log('peripheral_gap_pattern.md')

In [ ]:
log('growth_report.md')

## 2. Exploratory analysis
Hypothesis-neutral: we describe, we do not yet test. Every figure gets a one-line takeaway under it.

### 2.1 Distributions (data quality first, then levels and growth)
- Price levels by metro and year — histograms and box plots, log scale. *Is a log transform warranted?* (Skewness table in `growth_report.md` says yes.)
- Cumulative growth by period and metro — box plots / violins. *How wide is the spread within a metro compared with the gap between metros?*
- ZHVI and ZORI coverage over time and distance — a small multiple of the data-quality flags (the figure behind Section 1).

In [ ]:
# TODO 2.1

### 2.2 The gradient (SQ1)
- Scatter: cumulative growth against distance from the center, one panel per metro, columns = period (pre-2020, 2020–2022, 2023–2025). Fit line per panel with its slope annotated; same y-axis scale across metros.
- Slope by year (December YoY on distance), one line per metro: *does the slope change at 2020 and does it come back?*
- Same figure with the nearest-center distance for Dallas and Tampa, next to the city-hall version: *does the second center change the picture?*

In [ ]:
# TODO 2.2

### 2.3 Spatial structure
- Choropleth of growth (relative to the metro median) per metro and period, one shared diverging color scale.
- Mark the center(s). *Smooth rings (monocentric) or patches (polycentric)?*

In [ ]:
# TODO 2.3

### 2.4 Relationships (SQ2)
- Pre-pandemic work-from-home share (2015–2019) against 2019–2022 growth: binned scatter, pooled and within metro (demeaned). Show the unreliable-share ZIPs in a different marker.
- Correlation matrix of exposure, income, housing mix, distance, growth. *How much of the exposure signal is just income and distance?*

In [ ]:
# TODO 2.4

### 2.5 Trajectories (SQ3, exploratory)
- Index (Dec 2019 = 100) paths for all ZIPs as faint lines with metro medians on top; faceted by distance band.
- Optional: k-means on normalized paths with a silhouette plot; map the clusters. *Are the groups distinct or a continuum?*

In [ ]:
# TODO 2.5

### 2.6 Robustness views
- Full sample vs. flag-free sample (see `zip_flags.csv`), side by side.
- ZORI check on the fixed ZIP sample, stated as an inner/middle-metro check (see the ZORI limitation in Section 1).

In [ ]:
# TODO 2.6

## 3. Implications
TODO (brief): what the exploration suggests for the modeling that follows — which results look robust to the center definition, which depend on sample choices, where the data cannot answer the question (outer-ring rents, noisy exposure measures), and what we would do next.

## References
Data sources only; see `docs/SOURCES.md` for the complete list with URLs and retrieval dates.